In [1]:
import os
from arcgis.gis import GIS

### Logg på ArcGIS for å få tilgang til ArcGIS Online
* Du blir spurt om du vil bruke kontoen du logger på ArcGIS Pro med
* Hvis du svarer 'n' vil scriptet forsøke å bruke verdier i .env
* Hvis du ikke har .env file kan du fylle inn brukernavn og passord selv

TODO: forenkle

In [2]:
import getpass
from pathlib import Path
from dotenv import load_dotenv

# Finn .env i prioritert rekkefolge: notebooks-mappe, deretter arbeidskatalog.
env_candidates = [
    Path("notebooks") / ".env",
    Path.cwd() / ".env",
]
env_file = next((p for p in env_candidates if p.exists()), None)

if env_file is not None:
    load_dotenv(env_file)
    print(f"Lastet inn variabler fra {env_file.resolve()}")
else:
    print("Fant ingen .env-fil. Bruker env-variabler/prompt videre.")

# Spør om du vil bruke kontoen du er logget på ArcGIS Pro med
use_pro = input("Vil du bruke kontoen du er logget på ArcGIS Pro med? (ja/nei): ").lower().strip()

if use_pro in ["ja", "j", "yes", "y"]:
    try:
        gis = GIS("home")
        print("Bruker ArcGIS Pro innstillinger...")
    except Exception as e:
        print(f"Kunne ikke koble til ArcGIS Pro: {e}")
        gis = None
else:
    gis = None

# Hvis ikke ArcGIS Pro, bruk miljøvariabler eller prompt
if gis is None:
    print("\nBruker miljøvariabler eller prompt...")

    portal = os.getenv("AGOL_URL", "https://www.arcgis.com")
    username = os.getenv("AGOL_USERNAME") or input("AGOL brukernavn: ")
    password = os.getenv("AGOL_PASSWORD") or getpass.getpass("AGOL passord: ")

    gis = GIS(portal, username, password)

print(f">>> Logget inn som: {gis.users.me.username} ({gis.users.me.fullName})")
print(f">>> AGOL organisasjon / Portal: {gis.properties.get('urlKey')} ({gis.properties.get('name')})")


Lastet inn variabler fra C:\Users\KJEPET\repos\arcpy-utils-public\notebooks\.env

Bruker miljøvariabler eller prompt...
>>> Logget inn som: mdiradmin (Mdir Admin)
>>> AGOL organisasjon / Portal: miljodir-ekstern (Miljødirektoratet ekstern)


### function: load_github_module
TODO: forenkle tilgang til denne

In [ ]:
import importlib.util
import os
import urllib.request
from pathlib import Path

def load_github_module(
    module: str,
    owner: str,
    repo: str,
    reference: str | None = None,
    force_reload: bool = False,
):
    """
    Last inn én Python-modul fra et GitHub-repository inn i gjeldende sesjon.

    `reference` kan være en branch, tag eller commit-hash.

    :param module: Sti til modulfil i repositoryet, for eksempel "modules/hello_world.py".
    :param owner: GitHub-eier eller organisasjon.
    :param repo: Navn på GitHub-repository.
    :param reference: Branch, tag eller commit-hash. Standard er "main".
    :param force_reload: Last ned på nytt selv om cache-fil finnes.
    :return: Importert modulobjekt.
    """
    reference = reference or "main"

    # Bruk /arcgis/home i AGOL-notebooks, ellers fall tilbake til brukers hjemmemappe.
    cache_root = Path("/arcgis/home") if Path("/arcgis/home").exists() else Path.home()
    local_file = cache_root / "modules_cache" / reference[:7] / module
    local_file.parent.mkdir(parents=True, exist_ok=True)

    url = f"https://raw.githubusercontent.com/{owner}/{repo}/{reference}/{module}"
    if not local_file.exists() or force_reload:
        urllib.request.urlretrieve(url, str(local_file))

    module_name = module.split("/")[-1].removesuffix(".py")

    spec = importlib.util.spec_from_file_location(module_name, str(local_file))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Kunne ikke opprette import-spec for {local_file}")

    loaded_module = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(loaded_module)
    except Exception as exc:
        raise RuntimeError(
            f"Feil ved lasting av modul fra {url}. Lokal fil: {local_file}. Feil: {exc}"
        ) from exc

    return loaded_module

* Brukes for å legge lister med brukere (e-postadresser for oidc-brukere og brukernavn for agol-brukere) til grupper.
* Beskrivelse av funksjonalitet:
    * Eksisternde oidc-brukere i gruppa fjernes før de legges til ** bør heller sjekke om de finnes **
    * agol-brukere legges til (eksisterende beholdes) - dette skiller seg fra oidc-brukere
    * admin users og owner for gruppen beholdes
    * remove/add kjøres i batcher (batch_size) for å unngå API-grense per kall

In [18]:
# Sett force_reload=True for å tvangs-laste ned ny versjon fra GitHub (overstyrer cache).
mod = load_github_module(
    owner="miljodirektoratet",
    repo="arcpy-utils-public",
    reference="agol_legg_brukere_til_grupper",
    module="src/mdir_arcpy_utils_public/agol_add_users_to_group.py",
    force_reload=True,
)
print("Loaded module:", mod.__name__)

Loaded module: agol_add_users_to_group


In [17]:
# Dette er brukere som har e-post som brukernavn og er automatisk opprettet via MinSide
oidc_brukere = [
    "agund76@gmail.com",
    "anja@sallirnatur.no",
    "ariane@sallirnatur.no",
    "bendik@sallirnatur.no",
    "birgit.solberg@multiconsult.no",
    "christine.potsch@biota.no",
    "christopher.reppe@ramboll.no",
    "daniel.skoog@multiconsult.no",
    "elin@sallirnatur.no",
    "elisabeth@tyrinatur.no",
    "ombler@natres.no",
    "eric.ombler@naturrestaurering.no",
    "fredrik@natsam.no",
    "geir@bioreg.no",
    "gunnar@natsam.no",
    "iris@natsam.no",
    "jarle@natsam.no",
    "jonklepsland@gmail.com",
    "jorn.lokken@natres.no",
    "kaj-andreas@ecofact.no",
    "karen@tyrinatur.no",
    "kine@sallirnatur.no",
    "knut@ecofact.no",
    "konstanse@dokkadeltaet.no",
    "kristoffer.selvig@asplanviak.no",
    "larserik@biofokus.no",
    "lars.Joran@tyrinatur.no",
    "linn.eilertsen@biota.no",
    "madlaina@biofokus.no",
    "magnus@naturfokusas.no",
    "magnus@dokkadeltaet.no",
    "mari.brondbo.dahl@norconsult.com",
    "metteline@ecofact.no",
    "oda@sallirnatur.no",
    "oivind@biofokus.no",
    "oystein@sallirnatur.no",
    "paul@tyrinatur.no",
    "solveig.stralberg@wsp.com",
    "stine.svang@wsp.com",
    "tellnes@mfu.no",
    "hanssen@mfu.no",
    "vetle.lindgren@norconsult.com",
    "zoficimburova@gmail.com",
    "tonje.berland@natres.no",
    "tonje@natres.no",
    "vilde.murer@multiconsult.no",
]

# Dette er unike brukernavn for ArcGIS kontoer - enten i miljodirektoratet eller miljodir-ekstern
arcgis_brukere = [
    "kjepet_mdir",
    "bouvetadmin",
    "coandr_mdir",
]

# Navn på gruppen som skal inneholde brukere fra begge bruker-listene
gruppe_navn = "p-referansedata_kartverktoey"

# Settes til True når du bare vil teste scriptet uten å gjøre endringer.
dry_run = True

mod.add_users_to_group(
    gis=gis,
    oidc_brukere=oidc_brukere,
    arcgis_brukere=arcgis_brukere,
    gruppe_navn=gruppe_navn,
    dry_run=dry_run,
)

>>> AGOL organisasjon:            miljodir-ekstern
>>> OIDC-brukere input:           ['agund76@gmail.com', 'anja@sallirnatur.no', 'ariane@sallirnatur.no', 'bendik@sallirnatur.no', 'birgit.solberg@multiconsult.no', 'christine.potsch@biota.no', 'christopher.reppe@ramboll.no', 'daniel.skoog@multiconsult.no', 'elin@sallirnatur.no', 'elisabeth@tyrinatur.no', 'ombler@natres.no', 'eric.ombler@naturrestaurering.no', 'fredrik@natsam.no', 'geir@bioreg.no', 'gunnar@natsam.no', 'iris@natsam.no', 'jarle@natsam.no', 'jonklepsland@gmail.com', 'jorn.lokken@natres.no', 'kaj-andreas@ecofact.no', 'karen@tyrinatur.no', 'kine@sallirnatur.no', 'knut@ecofact.no', 'konstanse@dokkadeltaet.no', 'kristoffer.selvig@asplanviak.no', 'larserik@biofokus.no', 'lars.Joran@tyrinatur.no', 'linn.eilertsen@biota.no', 'madlaina@biofokus.no', 'magnus@naturfokusas.no', 'magnus@dokkadeltaet.no', 'mari.brondbo.dahl@norconsult.com', 'metteline@ecofact.no', 'oda@sallirnatur.no', 'oivind@biofokus.no', 'oystein@sallirnatur.no', 'pa

{}